# Part 2b: SP Scaling Study (All 5 Models)

SVG Scaling Laws Project

**Runtime**: A100 GPU

**Prerequisites**: Run `colab_lr_sweep.ipynb` first (data + SP LR sweep).

**Estimated time**: ~2.5 hours (Tiny 5m + Small 10m + Medium 20m + Large 30m + XL 80m)

## 1. Setup

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

%cd /content/drive/MyDrive/svg-scaling-project
!git pull
!pip install -r requirements.txt -q

In [ ]:
import torch
print(f"CUDA: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"Memory: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")

## 2. Determine Optimal LR from Sweep

In [ ]:
import json, os

sweep_dir = 'results/runs/lr_sweep'
best_loss = float('inf')
optimal_lr = None

print(f'{"Run":>20s}  {"LR":>10s}  {"Final Val Loss":>14s}')
print('-' * 50)
for run_name in sorted(os.listdir(sweep_dir)):
    summary_path = os.path.join(sweep_dir, run_name, 'summary.json')
    if not os.path.exists(summary_path):
        continue
    with open(summary_path) as f:
        s = json.load(f)
    vl = s['final_val_loss']
    run_lr = s['config']['training']['learning_rate']
    print(f'{run_name:>20s}  {run_lr:>10.0e}  {vl:>14.4f}')
    if vl < best_loss:
        best_loss = vl
        optimal_lr = run_lr

lr_str = f'{optimal_lr:.0e}'
print(f'\n\u2192 Optimal SP LR: {lr_str} (final_val_loss={best_loss:.4f})')

## 3. Train All 5 Models

In [ ]:
import subprocess, time

models = ['tiny', 'small', 'medium', 'large', 'xl']
results = []

for name in models:
    output_dir = f'results/runs/sp/{name}'
    print(f'\n{"=" * 60}')
    print(f'SP Scaling Study: {name} (LR={lr_str})')
    print(f'{"=" * 60}')
    t0 = time.time()

    result = subprocess.run([
        'python', 'src/train.py',
        '--config', f'configs/{name}.yaml',
        '--learning-rate', lr_str,
        '--output-dir', output_dir,
        '--device', 'cuda',
    ])

    elapsed = time.time() - t0
    status = 'OK' if result.returncode == 0 else f'FAIL (exit {result.returncode})'
    print(f'\n\u2192 {name}: {status}, {elapsed/60:.1f} min')

    summary_path = f'{output_dir}/summary.json'
    try:
        with open(summary_path) as f:
            s = json.load(f)
        results.append(s)
        print(f'  final_val_loss={s["final_val_loss"]:.4f}, final_ppl={s["final_val_ppl"]:.2f}')
    except FileNotFoundError:
        print(f'  (no summary.json)')

## 4. Results

In [ ]:
print(f'{"Model":>8s}  {"Params":>8s}  {"Final Val Loss":>14s}  {"Best Val Loss":>14s}  {"Final PPL":>10s}')
print('-' * 60)
for r in results:
    print(f'{r["config_name"]:>8s}  {r["n_params"]/1e6:>7.1f}M  '
          f'{r["final_val_loss"]:>14.4f}  {r["best_val_loss"]:>14.4f}  '
          f'{r["final_val_ppl"]:>10.2f}')

print(f'\nSP Scaling Study complete. Run analyze_scaling.py locally for power law fit.')